# ✅ Solution 1: Identifikasi Tipe Data

**Approach**: Analisis domain knowledge + verifikasi dengan pandas  
**Key Insight**: `dtypes` di Pandas hanya menunjukkan format penyimpanan, bukan tipe data statistik yang sebenarnya.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

data_ecommerce = {
    'order_id':          ['ORD-001','ORD-002','ORD-003','ORD-004','ORD-005','ORD-006','ORD-007','ORD-008'],
    'kategori_produk':   ['elektronik','fashion','makanan','elektronik','fashion','makanan','elektronik','fashion'],
    'harga_produk':      [1250000.0,350000.0,85000.0,3200000.0,125000.0,45000.0,875000.0,290000.0],
    'jumlah_item':       [1,2,5,1,3,10,1,2],
    'rating_penjual':    [4,5,3,5,4,2,5,3],
    'status_pengiriman': ['terkirim','proses','terkirim','terkirim','proses','terkirim','proses','terkirim'],
    'metode_bayar':      ['transfer','qris','cod','transfer','qris','cod','transfer','qris'],
    'berat_paket_kg':    [0.5,1.2,2.8,0.3,0.8,5.0,0.4,1.5],
    'kode_pos_tujuan':   [12345,60111,40115,10110,50132,80117,30123,70219],
    'sudah_review':      [True,False,True,True,False,True,False,True],
    'prioritas_ongkir':  ['reguler','express','same-day','reguler','express','reguler','same-day','express'],
    'diskon_persen':     [0,10,0,20,5,0,15,10],
}
df = pd.DataFrame(data_ecommerce)
df.head(3)

## 📖 Penjelasan Pendekatan

Untuk mengklasifikasikan tipe data, gunakan urutan pertanyaan:
1. Apakah nilainya bisa **dijumlah dengan makna**? → Jika ya, numerik
2. Apakah nilainya numerik tapi **hanya sebagai label/identifikasi**? → Nominal meski berupa angka
3. Apakah ada **urutan yang bermakna** antar kategori? → Ordinal
4. Apakah nilainya selalu **bilangan bulat**? → Discrete (bukan Continuous)

In [ ]:
# ============================================================
# SOLUSI TODO 1: Tabel Klasifikasi Tipe Data
# ============================================================
klasifikasi = {
    'kolom': [
        'order_id', 'kategori_produk', 'harga_produk', 'jumlah_item',
        'rating_penjual', 'status_pengiriman', 'metode_bayar',
        'berat_paket_kg', 'kode_pos_tujuan', 'sudah_review',
        'prioritas_ongkir', 'diskon_persen'
    ],
    'tipe_data': [
        'Nominal',     # order_id
        'Nominal',     # kategori_produk
        'Continuous',  # harga_produk
        'Discrete',    # jumlah_item
        'Ordinal',     # rating_penjual
        'Nominal',     # status_pengiriman
        'Nominal',     # metode_bayar
        'Continuous',  # berat_paket_kg
        'Nominal',     # kode_pos_tujuan  ← INI JEBAKAN
        'Nominal',     # sudah_review (binary nominal)
        'Ordinal',     # prioritas_ongkir
        'Continuous',  # diskon_persen (bisa juga Discrete, tapi konteks persen lebih ke continuous)
    ],
    'alasan': [
        'ID unik per order, hanya label — tidak bermakna dijumlah',
        'Kategori tanpa urutan: elektronik tidak > fashion',
        'Harga bisa bernilai desimal, bisa dijumlah dengan makna',
        'Hitungan bulat: tidak mungkin beli 1.5 item',
        'Ada urutan: 5 > 4 > 3 > 2, tapi jarak antar bintang tidak pasti sama',
        'Hanya 2 status: proses/terkirim — tidak ada urutan bermakna',
        'Kategori pembayaran tanpa urutan: qris tidak lebih baik dari cod',
        'Berat dalam kg, nilai riil/desimal, bisa dijumlah',
        'JEBAKAN: meski angka, kode pos adalah label wilayah — tidak bisa dijumlah/dirata-rata',
        'Hanya True/False — binary nominal, tidak ada urutan',
        'Ada urutan: same-day > express > reguler (kecepatan)',
        'Persentase bisa bernilai desimal; dalam konteks ini kontinu 0-100',
    ]
}

df_klas = pd.DataFrame(klasifikasi)
print('Klasifikasi Tipe Data E-Commerce:')
display(df_klas)

print()
print('Ringkasan:')
print(df_klas['tipe_data'].value_counts())

In [ ]:
# ============================================================
# SOLUSI TODO 2: Kolom yang Menipu
# ============================================================
kolom_menipu = ['kode_pos_tujuan', 'rating_penjual']
penjelasan = {
    'kode_pos_tujuan': 'Berupa int64 di pandas, tapi ini label wilayah. Rata-rata kode pos tidak bermakna.',
    'rating_penjual':  'Berupa int64, tapi ini Ordinal. Bisa diurutkan, tapi jarak 3→4 belum tentu sama dengan 4→5.',
}

print('Kolom yang Menipu (terlihat numerik, tapi bukan numerik statistik):')
for k, p in penjelasan.items():
    print(f'  {k}: {p}')
    print(f'    Pandas dtypes: {df[k].dtype}')
    print()

In [ ]:
# ============================================================
# SOLUSI TODO 3: Urutan Kolom Ordinal
# ============================================================
urutan_ordinal = {
    'rating_penjual':  [1, 2, 3, 4, 5],
    'prioritas_ongkir': ['reguler', 'express', 'same-day'],
}

print('Urutan Ordinal yang Benar:')
for kolom, urutan in urutan_ordinal.items():
    print(f'  {kolom}: {urutan} (dari terkecil ke terbesar)')
print()

# Demonstrasi: set tipe CategoricalDtype untuk prioritas_ongkir
ongkir_dtype = pd.CategoricalDtype(categories=['reguler','express','same-day'], ordered=True)
df['prioritas_ongkir'] = df['prioritas_ongkir'].astype(ongkir_dtype)
print('Setelah set sebagai Ordinal Categorical:')
print(df['prioritas_ongkir'].dtype)
print('Test: same-day > reguler?', df['prioritas_ongkir'][2] > df['prioritas_ongkir'][0])

## 📌 Takeaways

- **`df.dtypes` bukan tipe data statistik** — hanya format penyimpanan pandas
- **Angka belum tentu numerik**: kode pos, ID, rating bisa jadi nominal/ordinal
- **Domain knowledge adalah kunci** — pahami konteks bisnis untuk tentukan tipe data yang benar
- **Ordinal perlu urutan eksplisit** — gunakan `pd.CategoricalDtype(ordered=True)` agar pandas tahu urutannya
- Kesalahan klasifikasi tipe data → preprocessing salah → model yang buruk